# Data Integration Phase
Enriches data from silver lake and writes aggregated and enriched taxi_trips data as gold Delta table `integrated_taxi_trips`.


## 1. Configure Spark


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("integration-gold")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7e9e2fa9-996a-4154-a426-49fa76cb4146;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 110ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs


## 2. Load silver tables

Read silver delta tables.

In [27]:
from pyspark.sql import functions as F

trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

## 3. Hourly weather (NYC local)



In [28]:
hourly_weather = (
    weather
    .groupBy(
        F.col("observation_date").alias("pickup_date"),
        F.col("observation_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("temperature_c").alias("temperature_c"),
        F.avg("wind_speed_ms").alias("wind_speed_ms"),
    )
)

print(f"hourly_weather: {hourly_weather.count():,} hours")
hourly_weather.show(3, truncate=False)


hourly_weather: 8,784 hours
+-----------+-----------+-------------+------------------+
|pickup_date|pickup_hour|temperature_c|wind_speed_ms     |
+-----------+-----------+-------------+------------------+
|2024-08-23 |2          |22.2         |3.1111111111111107|
|2024-08-23 |7          |20.6         |3.611111111111111 |
|2024-08-23 |11         |18.9         |0.0               |
+-----------+-----------+-------------+------------------+
only showing top 3 rows



## 4. Hourly air quality (NYC local)



In [29]:
NYC_COUNTIES = [5, 47, 61, 81, 85]

hourly_aq = (
    air_quality
    .filter(F.col("county_code").isin(NYC_COUNTIES))
    .groupBy(
        F.col("measurement_date").alias("pickup_date"),
        F.col("measurement_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("value").alias("pm25"),
        F.first("unit").alias("pm25_unit"),
    )
)

print(f"hourly_aq: {hourly_aq.count():,} hours")
hourly_aq.show(3, truncate=False)


hourly_aq: 8,783 hours
+-----------+-----------+------------------+---------------------------+
|pickup_date|pickup_hour|pm25              |pm25_unit                  |
+-----------+-----------+------------------+---------------------------+
|2024-01-01 |0          |14.520000000000001|Micrograms/cubic meter (LC)|
|2024-01-01 |1          |14.459999999999999|Micrograms/cubic meter (LC)|
|2024-01-01 |2          |14.440000000000001|Micrograms/cubic meter (LC)|
+-----------+-----------+------------------+---------------------------+
only showing top 3 rows



## 5. Pickup / dropoff zone lookups

`taxi_zones` is a small table ==> we can easily and with minimal overhead split them into `pickup_zones` and `dropoff_zones`


In [30]:
pickup_zones = zones.select(
    F.col("location_id").alias("pickup_location_id"),
    F.col("zone").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough"),
)

dropoff_zones = zones.select(
    F.col("location_id").alias("dropoff_location_id"),
    F.col("zone").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough"),
)


## 6. Enrich trips and write `integrated_taxi_trips`

Left-join weather and air_quality data onto taxi_trip data.

In [31]:
integrated = (
    trips
    .join(F.broadcast(pickup_zones), "pickup_location_id", "left")
    .join(F.broadcast(dropoff_zones), "dropoff_location_id", "left")
    .join(F.broadcast(hourly_weather), ["pickup_date", "pickup_hour"], "left")
    .join(F.broadcast(hourly_aq), ["pickup_date", "pickup_hour"], "left")
    .fillna(
        {
            "pickup_zone": "UNKNOWN",
            "pickup_borough": "UNKNOWN",
            "dropoff_zone": "UNKNOWN",
            "dropoff_borough": "UNKNOWN",
        }
    )
    .select(
        "taxi_type",
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough",
        "dropoff_location_id",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "temperature_c",
        "wind_speed_ms",
        "pm25",
        "pm25_unit",
        "pickup_date",
        "pickup_hour",
    )
    .cache()
)

# Materialize the shared input once so both writes measure partitioning and I/O.
integrated.count()

import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips", partition_by=["pickup_date"])
print(f"write by_date: {time.perf_counter() - t0:.1f}s")
show_delta(spark, GOLD / "integrated_taxi_trips")

write by_date: 25.7s


integrated_taxi_trips: 9417383 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+---------------------+--------------+-------------------+-----------------------+---------------+-----------+----------+------------+------------+-------------+-----------------+------------------+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone          |pickup_borough|dropoff_location_id|dropoff_zone           |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms    |pm25              |pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+---------------------+--------------+-------------------+-----------------------+--------------

## 7. Two storage designs

To compare two storage designs, we can partition the same rows aggregated from silver to gold layer in two different ways:

| Partitioning Logic | Table | Partition | Suited for |
| --- | --- | --- | --- |
| By timestamp | `integrated_taxi_trips` | `pickup_date` | average duration per day |
| By location | `integrated_taxi_trips_by_borough` | `pickup_borough` | trip data per borough location |


In [32]:
import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips_by_borough", partition_by=["pickup_borough"])
print(f"write by_borough: {time.perf_counter() - t0:.1f}s")


def storage_report(table_name: str) -> None:
    path = GOLD / table_name
    files = [f for f in path.rglob("*.parquet") if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
    n_parts = len({f.parent for f in files})
    print(
        f"{table_name:40} files={len(files):>5}  partitions={n_parts:>4}  size={size_mb:>8.1f} MB"
    )


print()
print("Storage")
storage_report("integrated_taxi_trips")
storage_report("integrated_taxi_trips_by_borough")

write by_borough: 35.3s

Storage
integrated_taxi_trips                    files=  222  partitions=  92  size=   227.7 MB
integrated_taxi_trips_by_borough         files=  200  partitions=   8  size=   226.9 MB


### Queries on both designs


In [33]:
import time


def queries(df):
    duration_min = (
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    ) / 60.0
    return {
        "trips per borough": (
            df.groupBy("pickup_borough")
            .agg(F.count(F.lit(1)).alias("trips"))
            .orderBy(F.desc("trips"))
        ),
        "avg duration per day": (
            df.withColumn("duration_min", duration_min)
            .groupBy("pickup_date")
            .agg(F.avg("duration_min").alias("avg_duration_min"))
            .orderBy("pickup_date")
        ),
        "avg fare per borough": (
            df.groupBy("pickup_borough")
            .agg(F.avg("fare_amount").alias("avg_fare"))
            .orderBy("pickup_borough")
        ),
    }


def run_queries(table_name: str) -> None:
    spark.catalog.clearCache()
    df = read_delta(spark, GOLD / table_name)
    print(f"\n{table_name}")
    print("-" * 40)
    for name, q in queries(df).items():
        t0 = time.perf_counter()
        rows = q.collect()
        elapsed = time.perf_counter() - t0
        print(f"\n{name}  ({elapsed:.2f}s, {len(rows):,} rows)")
        spark.createDataFrame(rows).show(20, truncate=False)


run_queries("integrated_taxi_trips")
run_queries("integrated_taxi_trips_by_borough")



integrated_taxi_trips
----------------------------------------



trips per borough  (2.37s, 8 rows)


+--------------+-------+
|pickup_borough|trips  |
+--------------+-------+
|Manhattan     |8442728|
|Queens        |817017 |
|Brooklyn      |95738  |
|Unknown       |31298  |
|Bronx         |24715  |
|N/A           |4752   |
|EWR           |915    |
|Staten Island |220    |
+--------------+-------+




avg duration per day  (9.31s, 92 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2024-01-01 |16.444424507823307|
|2024-01-02 |16.906788016430834|
|2024-01-03 |16.545877362397537|
|2024-01-04 |16.08400669921135 |
|2024-01-05 |15.385487487153334|
|2024-01-06 |14.58333002828419 |
|2024-01-07 |13.901090108635216|
|2024-01-08 |15.62057394556443 |
|2024-01-09 |15.068740392765223|
|2024-01-10 |15.230732569936643|
|2024-01-11 |16.433418924474225|
|2024-01-12 |16.471563084834628|
|2024-01-13 |15.074412049124728|
|2024-01-14 |14.352769278119478|
|2024-01-15 |14.90966166434527 |
|2024-01-16 |16.602465760869592|
|2024-01-17 |16.343770248792744|
|2024-01-18 |16.05834059067091 |
|2024-01-19 |15.139623995861996|
|2024-01-20 |14.266279737782696|
+-----------+------------------+
only showing top 20 rows




avg fare per borough  (1.55s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+------------------+
|Bronx         |31.733360712118152|
|Brooklyn      |28.560865487058393|
|EWR           |90.21594535519125 |
|Manhattan     |15.430819608307019|
|N/A           |87.23950968013462 |
|Queens        |51.962818815275156|
|Staten Island |42.360727272727274|
|Unknown       |20.224147868873448|
+--------------+------------------+


integrated_taxi_trips_by_borough
----------------------------------------



trips per borough  (2.89s, 8 rows)
+--------------+-------+
|pickup_borough|trips  |
+--------------+-------+
|Manhattan     |8442728|
|Queens        |817017 |
|Brooklyn      |95738  |
|Unknown       |31298  |
|Bronx         |24715  |
|N/A           |4752   |
|EWR           |915    |
|Staten Island |220    |
+--------------+-------+




avg duration per day  (2.08s, 92 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2024-01-01 |16.44442450782354 |
|2024-01-02 |16.906788016430923|
|2024-01-03 |16.54587736239775 |
|2024-01-04 |16.08400669921109 |
|2024-01-05 |15.38548748715337 |
|2024-01-06 |14.583330028284262|
|2024-01-07 |13.901090108635143|
|2024-01-08 |15.620573945564523|
|2024-01-09 |15.068740392765537|
|2024-01-10 |15.23073256993671 |
|2024-01-11 |16.433418924474434|
|2024-01-12 |16.47156308483458 |
|2024-01-13 |15.074412049124797|
|2024-01-14 |14.352769278119418|
|2024-01-15 |14.90966166434513 |
|2024-01-16 |16.6024657608695  |
|2024-01-17 |16.34377024879265 |
|2024-01-18 |16.058340590670788|
|2024-01-19 |15.139623995861927|
|2024-01-20 |14.266279737782732|
+-----------+------------------+
only showing top 20 rows


avg fare per borough  (0.81s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+---------------

26/09/11 16:29:08 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 566556 ms exceeds timeout 120000 ms
26/09/11 16:29:08 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/11 16:29:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

### Query 3

In [35]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_3 = """
WITH rounded_pm25 AS (
    SELECT
        ROUND(pm25, 0) AS pm25_level,
        pickup_date,
        pickup_hour
    FROM integrated_taxi_trips
    WHERE pm25 IS NOT NULL
      AND pickup_date IS NOT NULL
      AND pickup_hour IS NOT NULL
)
SELECT
    pm25_level,
    COUNT(*) AS trips,
    COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS observed_hours,
    ROUND(
        CAST(COUNT(*) AS DOUBLE)
        / COUNT(DISTINCT struct(pickup_date, pickup_hour)),
        2
    ) AS trips_per_hour
FROM rounded_pm25
GROUP BY pm25_level
ORDER BY pm25_level DESC
"""

pm25_demand = spark.sql(query_3)
pm25_demand.show(34, truncate=False)

+----------+-------+--------------+--------------+
|pm25_level|trips  |observed_hours|trips_per_hour|
+----------+-------+--------------+--------------+
|34.0      |4188   |1             |4188.0        |
|33.0      |9071   |2             |4535.5        |
|31.0      |5019   |1             |5019.0        |
|30.0      |18438  |3             |6146.0        |
|29.0      |18240  |2             |9120.0        |
|28.0      |17702  |3             |5900.67       |
|27.0      |14835  |2             |7417.5        |
|26.0      |66047  |10            |6604.7        |
|25.0      |18435  |3             |6145.0        |
|24.0      |54765  |10            |5476.5        |
|23.0      |20069  |5             |4013.8        |
|22.0      |54775  |12            |4564.58       |
|21.0      |80355  |17            |4726.76       |
|20.0      |83362  |16            |5210.13       |
|19.0      |79234  |17            |4660.82       |
|18.0      |132230 |31            |4265.48       |
|17.0      |105092 |29         

### Query 4

In [30]:
query = """
WITH trips_with_weather AS (
    SELECT 
        pickup_zone,
        pickup_date,
        pickup_hour,
        NTILE(4) OVER (ORDER BY (10 * sqrt(wind_speed_ms) - wind_speed_ms + 10.5) * (33 - temperature_c)) AS weather_condition
    FROM integrated_taxi_trips
    WHERE pickup_zone IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
),
hourly_demand AS (
    SELECT 
        pickup_zone,
        weather_condition,
        COUNT(1) / COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS trips_per_hour
    FROM trips_with_weather
    GROUP BY pickup_zone, weather_condition
),
pivoted AS (
    SELECT * FROM hourly_demand
    PIVOT (
        ROUND(AVG(trips_per_hour), 2)
        FOR weather_condition IN (1 AS coldest, 2 AS cool, 3 AS warm, 4 AS warmest)
    )
)
SELECT 
    pickup_zone,
    coldest, cool, warm, warmest,
    ROUND(((GREATEST(coldest, cool, warm, warmest) - LEAST(coldest, cool, warm, warmest)) / ((coldest + cool + warm + warmest) / 4.0)) * 100, 2) AS pct_variation
FROM pivoted
WHERE (coldest + cool + warm + warmest) / 4.0 >= 10
ORDER BY pct_variation DESC
"""

spark.sql(query).show(truncate=False)

26/09/18 16:00:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/18 16:00:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/18 16:00:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/18 16:00:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/18 16:00:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/18 16:00:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/18 16:00:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/18 16:00:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------------------------+-------+------+------+-------+-------------+
|pickup_zone                  |coldest|cool  |warm  |warmest|pct_variation|
+-----------------------------+-------+------+------+-------+-------------+
|Financial District South     |16.16  |12.75 |12.32 |10.74  |41.72        |
|Financial District North     |26.19  |21.21 |20.93 |18.38  |36.03        |
|Meatpacking/West Village West|48.99  |38.88 |38.5  |34.97  |34.76        |
|Lower East Side              |57.45  |45.83 |45.13 |42.61  |31.08        |
|Little Italy/NoLiTa          |52.17  |41.67 |42.42 |38.75  |30.67        |
|Greenwich Village South      |75.67  |61.92 |63.01 |57.4   |28.33        |
|West Village                 |121.01 |99.43 |101.78|92.37  |27.63        |
|TriBeCa/Civic Center         |66.63  |58.62 |55.86 |51.16  |26.64        |
|East Village                 |123.21 |102.46|103.58|94.98  |26.62        |
|Battery Park City            |31.01  |28.21 |26.69 |24.06  |25.28        |
|World Trade